In [1]:
import pandas as pd
import numpy as np

# 1. Load Data
df = pd.read_csv('Atlantic_United_States.csv')

# 2. Data Validation & Formatting
# Convert date string to actual datetime objects for time-series analysis
df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y') 

# Convert duration from milliseconds to minutes
df['duration_min'] = df['duration_ms'] / 60000

# 3. Feature Engineering: Song-Level Metrics
# Grouping by both song and artist to calculate longevity and rank stats
song_metrics = df.groupby(['song', 'artist']).agg(
    days_on_chart=('date', 'count'),
    avg_rank=('position', 'mean'),
    best_rank=('position', 'min'),
    rank_volatility=('position', 'std') # Standard deviation measures volatility
).reset_index()

# If a song is only on the chart for 1 day, standard deviation is NaN. Fill with 0.
song_metrics['rank_volatility'] = song_metrics['rank_volatility'].fillna(0)

# 4. Feature Engineering: Popularity Trend Score (7-Day Rolling Average)
# Sort by song and date to ensure time is sequential
df = df.sort_values(by=['song', 'date'])

# Calculate rolling average for each song
df['pop_trend_7d'] = df.groupby('song')['popularity'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)

# 5. Artist Dominance Metric
# Total number of days an artist appears on the chart across all their songs
artist_dominance = df.groupby('artist').agg(
    total_chart_days=('date', 'count'),
    unique_songs=('song', 'nunique')
).reset_index().sort_values(by='total_chart_days', ascending=False)

# Display the first few rows of our new metrics
print("Song Metrics Snapshot:")
display(song_metrics.head())

Song Metrics Snapshot:


,song,artist,days_on_chart,avg_rank,best_rank,rank_volatility
0,"""Slut!"" (Taylor's Version) (From The Vault)",Taylor Swift,26,15.500000,3,12.130128
1,(Don't Fear) The Reaper,Blue Öyster Cult,1,50.000000,50,0.000000
2,(There's No Place Like) Home for the Holidays ...,Perry Como,2,45.000000,45,0.000000
3,16 CARRIAGES,Beyoncé,10,26.000000,16,10.635371
4,2 hands,Tate McRae,7,33.571429,8,12.286268
